### Explicación del script ETL.py (Extracción y Limpieza de Datos)

Este archivo es el encargado de la primera fase del proyecto: ingerir los datos crudos, evaluar su calidad y aplicar las transformaciones necesarias para tener una base sólida y confiable. Está estructurado en dos etapas principales:

- Etapa 1: Análisis y Calidad de Datos

Inspección Estructural (Eu): La función etapa_eu_calidad_datos se encarga de leer el archivo CSV original. Su objetivo es generar un diagnóstico rápido utilizando funciones de Pandas como .head(), .info(), y .describe(). Esto nos permite conocer el volumen de los datos (filas y columnas), los tipos de datos detectados y estadísticas descriptivas iniciales.

Detección de Anomalías (Jonathan): La función etapa_jonathan_calidad_datos complementa el análisis buscando registros duplicados exactos e imprimiendo el conteo de valores de columnas categóricas clave. Esto es vital para identificar inconsistencias tipográficas (ej. variaciones en mayúsculas/minúsculas o espacios adicionales) antes de limpiar.

- Etapa 2: Limpieza de Datos

Transformación (Eu): A través de etapa_eu_limpieza, se ejecutan las acciones correctivas. Primero se eliminan los duplicados detectados. Luego, se itera sobre todas las columnas de texto aplicando .str.strip() para eliminar espacios en blanco accidentales al inicio o final de las palabras, estandarizando así las categorías.

Validación (Jonathan): Finalmente, etapa_jonathan_validacion_final actúa como un control de calidad. Compara la cantidad de registros y columnas del dataset original contra el dataset procesado para asegurar que no hubo pérdida crítica de información durante la limpieza.

Salida: El script concluye exportando el resultado como dataset_limpio.csv, el cual servirá como base para el modelado de datos.

### Explicación del script star_schema.py (Modelado de Datos)

Una vez que los datos están limpios, este script transforma la estructura de un archivo plano a un modelo analítico de estrella (Star Schema). Este modelo divide la información en "Dimensiones" (tablas de catálogo) y una "Tabla de Hechos" central, optimizando la base para consultas en herramientas de Business Intelligence.

- Etapa 3 y 4: Construcción de Dimensiones
El trabajo se dividió temáticamente para extraer los valores únicos de cada categoría, generarles un identificador numérico (ID) y exportarlos como tablas independientes:

Dimensiones Demográficas (Eu): Crea catálogos para los perfiles de los encuestados (Dim_Edad, Dim_Genero, Dim_Departamento, Dim_Municipio, Dim_Ocupacion).

Dimensiones de Marketing (Jonathan): Crea catálogos para los motivadores de compra (Dim_Seleccion, Dim_Jugador, Dim_Promocion).

Dimensiones de Consumo y Tiempo (Nicole): Genera catálogos de los hábitos (Dim_Frecuencia, Dim_LugarCompra, etc.) y la tabla Dim_Tiempo para análisis cronológicos.

Construcción de la Tabla de Hechos (Nicole)
La función construir_tabla_hechos es el núcleo del modelo. Toma el dataset_limpio.csv y realiza uniones (usando merge o "JOINs") con cada una de las tablas dimensionales creadas anteriormente. El objetivo es reemplazar los textos descriptivos por sus respectivos IDs numéricos.

Salida: El resultado es Fact_Encuestas.csv, una tabla transaccional altamente eficiente compuesta casi en su totalidad por llaves foráneas numéricas y métricas directas (como el precio aceptado).

### 3. Explicación del script dashboard.py (Métricas y Visualizaciones)

Este último bloque de código se encarga de consumir los datos procesados para generar insights visuales a través de las librerías Matplotlib y Seaborn. Las visualizaciones se guardan automáticamente como imágenes (PNG) para su integración en reportes o dashboards.

- Métricas Demográficas (Eu)
La función generar_graficos_demograficas calcula el KPI principal (total de encuestas) y genera visualizaciones sobre quiénes son los encuestados. Utiliza gráficos de pastel para distribuciones simples (género) y gráficos de barras (verticales y horizontales) ordenados lógicamente para representar la edad y la concentración geográfica (departamentos y top 10 municipios).

- Métricas de Marketing (Jonathan)
La función generar_graficos_marketing visualiza las intenciones de compra y preferencias publicitarias. Un paso clave en este script ocurre antes de graficar: la columna de Jugadores (que contenía múltiples valores separados por punto y coma) se divide y se procesa con la función .explode(). Esto permite separar a los jugadores en filas individuales para que el conteo de popularidad sea exacto.

Se utilizan histogramas para visualizar la distribución del presupuesto (Precio_Aceptado).

Se utilizan gráficos de barras para crear los rankings del Top 10 de Selecciones y Jugadores que más influyen en la compra.

Se utilizan gráficos de dona/pastel y barras para las preferencias de promociones y canales publicitarios.

In [ ]:
import pandas as pd
from IPython.display import display, HTML
import os

# Ruta donde se encuentran las tablas limpias (Ajusta la ruta según tus carpetas)
ruta_tablas = "../3_Tablas_Limpias/" 

# Diccionario con el archivo y su título con iconos
tablas_nombradas = {
    "1. Dimensión Edad": "Dim_Edad.csv",
    "2. Dimensión Género": "Dim_Genero.csv",
    "3. Dimensión Departamento": "Dim_Departamento.csv",
    "4. Dimensión Municipio": "Dim_Municipio.csv",
    "5. Dimensión Ocupación": "Dim_Ocupacion.csv",
    "6. Dimensión Frecuencia": "Dim_Frecuencia.csv",
    "7. Dimensión Lugar de Compra": "Dim_LugarCompra.csv",
    "8. Dimensión Compañía Partidos": "Dim_CompaniaPartidos.csv",
    "9. Dimensión Snacks": "Dim_Snacks.csv",
    "10. Dimensión Sabor": "Dim_Sabor.csv",
    "11. Dimensión Presentación": "Dim_Presentacion.csv",
    "12. Dimensión Tiempo": "Dim_Tiempo.csv",
    "13. Dimensión Selección": "Dim_Seleccion.csv",
    "14. Dimensión Jugador": "Dim_Jugador.csv",
    "15. Dimensión Promoción": "Dim_Promocion.csv",
    "16. TABLA DE HECHOS CENTRAL (Fact_Encuestas)": "Fact_Encuestas.csv"
}

# CSS para darle diseño profesional a las tablas de Pandas
estilos_tabla = [
    # Diseño para los encabezados (Nombres de las columnas)
    {'selector': 'th', 
     'props': [('background-color', '#1F618D'), 
               ('color', 'white'), 
               ('font-family', 'Segoe UI, Arial, sans-serif'),
               ('font-weight', 'bold'),
               ('text-align', 'center'), 
               ('padding', '10px 15px'),
               ('border', '1px solid #1A5276')]},
    # Diseño para las celdas de datos
    {'selector': 'td', 
     'props': [('font-family', 'Segoe UI, Arial, sans-serif'), 
               ('padding', '8px 15px'), 
               ('text-align', 'center'), 
               ('border', '1px solid #D6EAF8')]},
    # Efecto Hover (Cambia de color al pasar el mouse por encima de la fila)
    {'selector': 'tr:hover', 
     'props': [('background-color', '#EBF5FB')]}
]

# Iterar sobre el diccionario para mostrar el título estilizado y la tabla
for titulo, archivo in tablas_nombradas.items():
    ruta_archivo = os.path.join(ruta_tablas, archivo)
    
    if os.path.exists(ruta_archivo):
        # 1. TÍTULO DE LA TABLA (Diseño tipo Tarjeta)
        display(HTML(f"""
        <div style='background: linear-gradient(90deg, #EBF5FB 0%, #ffffff 100%); 
                    padding: 12px 15px; 
                    border-left: 6px solid #2980B9; 
                    border-radius: 4px;
                    margin-top: 30px; 
                    margin-bottom: 15px;
                    box-shadow: 2px 2px 5px rgba(0,0,0,0.05);'>
            <h3 style='margin: 0; color: #1A5276; font-family: Segoe UI, Arial, sans-serif;'>
                {titulo} <span style='font-size: 14px; color: #7F8C8D; font-weight: normal;'>(Archivo: {archivo})</span>
            </h3>
        </div>
        """))
        
        # 2. LECTURA Y ESTILIZADO DE LA TABLA PANDAS
        df_temp = pd.read_csv(ruta_archivo)
        
        # Aplicamos el CSS, ocultamos el índice (0,1,2,3...) y mostramos los primeros 5 renglones
        tabla_estilizada = (df_temp.head(5)
                            .style.set_table_styles(estilos_tabla)
                            .hide(axis='index')) # Esto es clave para que se vea más limpia
        
        display(tabla_estilizada)
        
        # 3. PIE DE TABLA (Resumen de registros)
        display(HTML(f"""
        <p style='color: #7F8C8D; font-size: 12px; font-family: Segoe UI, Arial, sans-serif; margin-top: 5px; margin-bottom: 30px;'>
            <em>✓ La tabla completa contiene <b>{len(df_temp)} registros únicos</b> listos para integrarse al Data Warehouse.</em>
        </p>
        """))
        
    else:
        # Mensaje de error estilizado por si falta algún CSV
        display(HTML(f"""
        <div style='background-color: #FDEDEC; padding: 10px; border-left: 5px solid #E74C3C; margin-top: 20px;'>
            <p style='margin: 0; color: #C0392B;'><b>Alerta de Archivo:</b> No se encontró la tabla <code>{archivo}</code> en la ruta especificada.</p>
        </div>
        """))

### 1. Tabla: Dim_Edad (Dimensión de Edad)
La tabla Dim_Edad es el catálogo central diseñado para segmentar generacionalmente a la base de consumidores. En la arquitectura del Star Schema, esta tabla aísla la variable de la edad asignándole una llave primaria (ID_Edad) que se conecta con la tabla de hechos. Esto evita procesar largas cadenas de texto repetitivas (como "18 - 24 años" o "Menos de 18 años") miles de veces en la memoria, optimizando el rendimiento del modelo Zero-ETL.

### 2. Tabla: Dim_Genero (Dimensión de Género)
La tabla Dim_Genero estandariza la identidad demográfica de los encuestados a través de su identificador único (ID_Genero) asociado a los valores categóricos (Masculino, Femenino, Prefiero no indicar). La creación de esta tabla fue posible gracias a las fases previas de limpieza de datos lideradas por Eu, donde se eliminaron inconsistencias tipográficas para asegurar que el catálogo fuera perfectamente único y agrupara los datos de forma exacta, sin bifurcaciones erróneas.

### 3. Tabla: Dim_Departamento (Dimensión de Departamento)
La tabla Dim_Departamento representa la segmentación geográfica a nivel macro. Este catálogo asigna un ID_Departamento a cada uno de los departamentos de Guatemala presentes en la muestra de los 2,500 encuestados. Estructuralmente, separar la ubicación en su propia dimensión facilita enormemente la creación de mapas coropléticos (mapas de calor geográficos) en herramientas de visualización como Power BI, ya que el motor analítico solo necesita leer el catálogo una vez para renderizar las zonas.

### 4. Tabla: Dim_Municipio (Dimensión de Municipio)
Complementando la visión macro, la tabla Dim_Municipio ofrece la granularidad geográfica a nivel micro. Este catálogo (ID_Municipio y Municipio) desciende un nivel más en la jerarquía espacial, permitiendo aislar ciudades, cabeceras municipales o localidades específicas. En el esquema de estrella, mantener los municipios separados en su propia tabla dimensional permite hacer filtros dinámicos muy rápidos sin sobrecargar la tabla de transacciones central.

### 5. Tabla: Dim_Ocupacion (Dimensión de Ocupación)
La tabla Dim_Ocupacion clasifica el estado laboral o profesional de los encuestados (ej. Estudiante, Profesional independiente, Empleado, Jubilado) otorgándole a cada categoría su respectivo ID_Ocupacion. Esta fue una de las tablas que requirió mayor limpieza previa, para unificar criterios. Analíticamente, es una dimensión socioeconómica que ayuda a perfilar el estilo de vida del consumidor de manera mucho más profunda que la edad por sí sola.

### 6. Tabla: Dim_Frecuencia (Dimensión de Frecuencia de Consumo)
La tabla Dim_Frecuencia aísla y estandariza los hábitos de compra repetitiva de los consumidores (ej. "Diario", "1 a 2 veces por semana", "Rara vez"). En el modelo Star Schema, esta dimensión permite reemplazar largas descripciones de texto por un simple ID_Frecuencia, optimizando el espacio de almacenamiento y acelerando las consultas cuando se agrupan grandes volúmenes de datos.

 ### 7. Tabla: Dim_LugarCompra (Dimensión de Lugar de Compra)
El catálogo Dim_LugarCompra agrupa los canales físicos y digitales donde el consumidor prefiere adquirir el producto (ej. "Supermercados", "Tiendas de barrio", "Aplicación de delivery"). Al estandarizar estos valores bajo un ID_LugarCompra, el equipo de analítica elimina el riesgo de tener categorías duplicadas por errores de escritura, asegurando agrupamientos perfectos en las gráficas de Power BI o Python.

### 8. Tabla: Dim_CompaniaPartidos (Dimensión de Compañía)
La tabla Dim_CompaniaPartidos cataloga el entorno social del encuestado al momento de ver un partido de fútbol (ej. "Solo", "Con amigos", "Con familia"). Técnicamente, es una dimensión pequeña pero vital para cruzarla con los tamaños de empaque y generar insights combinados en la tabla de hechos.

### 9. Tabla: Dim_Snacks (Dimensión de Snacks Actuales)
La dimensión Dim_Snacks actúa como el catálogo de inventario base, unificando los nombres de los productos preferidos de la marca (ej. "CrunchMax BBQ", "Gol Nachos Supreme"). Esta tabla requirió un trabajo especial de partición, ya que los encuestados podían seleccionar múltiples opciones a la vez. Al separarlo en un catálogo único, evitamos violar las reglas de normalización de datos.

### 10. Tabla: Dim_Sabor (Dimensión de Perfil de Sabor)
El catálogo Dim_Sabor estandariza las preferencias gustativas del mercado, clasificando opciones como "Picante", "Salado", "Dulce" o "Queso". En el esquema relacional, permite cruzar rápidamente perfiles demográficos (ej. edad o municipio) con preferencias de paladar, utilizando el ID_Sabor.


### 11. Tabla: Dim_Presentacion (Dimensión de Presentación/Empaque)
La dimensión Dim_Presentacion categoriza el formato físico del producto que el usuario compra regularmente (ej. "Individual", "Familiar", "Megapack"). Al contar con su propia tabla, el modelo analítico se mantiene limpio y permite análisis rápidos sobre márgenes de ganancia.

### 12. Tabla: Dim_Tiempo (Dimensión Cronológica)
La tabla Dim_Tiempo aísla las fechas en las que se recolectó la información de la encuesta (ej. desglosando Año, Mes, Día si es necesario). En cualquier modelo Star Schema corporativo, la dimensión de tiempo es obligatoria, ya que todas las herramientas de Business Intelligence la necesitan para generar inteligencia de tiempo (Time Intelligence), como comparativas de mes a mes o análisis de tendencias.

### 13. Tabla: Dim_Seleccion (Dimensión de Selección)
La tabla Dim_Seleccion actúa como el catálogo definitivo de las selecciones nacionales de fútbol que los consumidores guatemaltecos planean apoyar durante el Mundial 2026 (ej. Argentina, Brasil, Francia). A nivel de arquitectura de datos, esta dimensión extrae las preferencias del texto crudo y les asigna un identificador único (ID_Seleccion), permitiendo que el motor analítico filtre y agrupe rápidamente los datos sin lidiar con errores tipográficos en la tabla central.

### 14. Tabla: Dim_Jugador (Dimensión de Jugador)
El catálogo Dim_Jugador aísla de manera individual a las estrellas del fútbol mundial que ejercen mayor influencia sobre la decisión de compra del consumidor (ej. Lionel Messi, Kylian Mbappé, Jude Bellingham). Debido a que los encuestados podían seleccionar múltiples jugadores en el formulario, la creación de esta tabla requirió un proceso técnico de desanidamiento (función explode) para garantizar que la base de datos cumpliera con las estrictas reglas de normalización del modelo dimensional, asignando a cada figura un ID_Jugador único.

### 15. Tabla: Dim_Promocion (Dimensión de Promoción)
La tabla Dim_Promocion clasifica y estandariza las mecánicas de enganche publicitario que el consumidor prefiere al interactuar con productos de temporada (ej. Tarjetas Coleccionables, Descuentos Directos, 2x1, Sorteos de Viajes). Al convertir estas estrategias en una dimensión estructurada mediante un ID_Promocion, el modelo de Business Intelligence puede evaluar rápidamente qué tipo de incentivo genera la mayor intención teórica de compra dentro del esquema de estrella.

### 16. Tabla: Fact_Encuestas (Tabla de Hechos Central)
La Fact_Encuestas es la obra maestra de la arquitectura del proyecto y el núcleo del Star Schema. Esta tabla ya no contiene textos descriptivos (como "Guatemala" o "Masculino"); está compuesta casi en su totalidad por llaves foráneas (ID_Edad, ID_Genero, ID_Frecuencia, etc.) que apuntan a las dimensiones, junto con las métricas transaccionales directas, como el presupuesto en quetzales (PrecioAdecuado).